In [ ]:
# Notebook 04 — PredRNN with Reverse Scheduled Sampling
# Phase 1: Wind Field Prediction (PredRNN)

# =====================================================
# 1. Imports and setup
# =====================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# =====================================================
# 2. PredRNN Cell (ST-LSTM with Memory Decoupling)
# =====================================================
class PredRNNCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size):
        super().__init__()
        padding = kernel_size // 2
        self.hidden_dim = hidden_dim

        self.conv_xh = nn.Conv2d(input_dim + hidden_dim, 4 * hidden_dim, kernel_size, padding=padding)
        self.conv_m = nn.Conv2d(hidden_dim, 3 * hidden_dim, kernel_size, padding=padding)

    def forward(self, x, h_prev, c_prev, m_prev):
        combined = torch.cat([x, h_prev], dim=1)
        gates = self.conv_xh(combined)
        i, f, g, o = torch.chunk(gates, 4, dim=1)

        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)

        c = f * c_prev + i * g

        m_gates = self.conv_m(m_prev)
        mi, mf, mg = torch.chunk(m_gates, 3, dim=1)
        mi = torch.sigmoid(mi)
        mf = torch.sigmoid(mf)
        mg = torch.tanh(mg)

        m = mf * m_prev + mi * mg

        h = o * torch.tanh(c + m)
        return h, c, m

# =====================================================
# 3. PredRNN Network
# =====================================================
class PredRNN(nn.Module):
    def __init__(self, input_dim, hidden_dims, kernel_size, num_layers, out_steps):
        super().__init__()
        self.num_layers = num_layers
        self.out_steps = out_steps

        self.cells = nn.ModuleList([
            PredRNNCell(input_dim if i == 0 else hidden_dims[i-1], hidden_dims[i], kernel_size)
            for i in range(num_layers)
        ])

        self.conv_out = nn.Conv2d(hidden_dims[-1], input_dim, kernel_size=1)

    def forward(self, x, teacher_forcing_ratio=0.0):
        # x: [B, Tin, C, H, W]
        B, Tin, C, H, W = x.shape
        device = x.device

        h = [torch.zeros(B, cell.hidden_dim, H, W, device=device) for cell in self.cells]
        c = [torch.zeros_like(h_i) for h_i in h]
        m = [torch.zeros_like(h_i) for h_i in h]

        outputs = []
        prev = x[:, -1]

        for t in range(self.out_steps):
            inp = prev
            for l, cell in enumerate(self.cells):
                h[l], c[l], m[l] = cell(inp, h[l], c[l], m[l])
                inp = h[l]
            frame = self.conv_out(inp)
            outputs.append(frame)

            if np.random.rand() < teacher_forcing_ratio:
                prev = x[:, -1]  # ground truth (simplified)
            else:
                prev = frame.detach()

        return torch.stack(outputs, dim=1)

# =====================================================
# 4. Scheduled Sampling Strategy (Normal - High to Low)
# =====================================================
def scheduled_sampling(epoch, max_epochs, start_ratio=0.9, end_ratio=0.1):
    """
    Early epochs: high teacher forcing (easy mode - learn correct patterns)
    Late epochs: low teacher forcing (hard mode - learn to predict independently)
    Linearly decreases from start_ratio to end_ratio
    """
    if max_epochs == 1:
        return start_ratio
    ratio = start_ratio - (start_ratio - end_ratio) * (epoch - 1) / (max_epochs - 1)
    return max(end_ratio, ratio)

# =====================================================
# 5. Training Loop
# =====================================================
def train_predrnn(model, train_loader, val_loader, epochs=20, lr=1e-3):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        tf_ratio = scheduled_sampling(epoch, epochs, start_ratio=0.9, end_ratio=0.1)
        train_loss = 0

        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            preds = model(x, teacher_forcing_ratio=tf_ratio)
            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                preds = model(x, teacher_forcing_ratio=0.0)
                val_loss += criterion(preds, y).item()

        val_loss /= len(val_loader)

        print(f"Epoch {epoch}/{epochs} | TF ratio: {tf_ratio:.2f} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

# =====================================================
# 6. Visualization Helper
# =====================================================
def plot_prediction(target, pred, channel=0):
    plt.plot(target[:, channel], label='Target')
    plt.plot(pred[:, channel], label='Predicted')
    plt.legend()
    plt.title('PredRNN Wind Prediction')
    plt.show()



Using device: cpu


In [ ]:
DATA_DIR = "data/processed"

X_train = torch.from_numpy(np.load(f"{DATA_DIR}/X_train.npy")).float()
Y_train = torch.from_numpy(np.load(f"{DATA_DIR}/Y_train.npy")).float()

X_val = torch.from_numpy(np.load(f"{DATA_DIR}/X_val.npy")).float()
Y_val = torch.from_numpy(np.load(f"{DATA_DIR}/Y_val.npy")).float()

X_test = torch.from_numpy(np.load(f"{DATA_DIR}/X_test.npy")).float()
Y_test = torch.from_numpy(np.load(f"{DATA_DIR}/Y_test.npy")).float()

print("Train:", X_train.shape, Y_train.shape)
print("Val:", X_val.shape, Y_val.shape)
print("Test:", X_test.shape, Y_test.shape)

# Verify normalization (should be ~0 mean, ~1 std)
print(f"\nData stats (should be ~0 mean, ~1 std if normalized):")
print(f"X_train mean: {X_train.mean().item():.4f}, std: {X_train.std().item():.4f}")
print(f"Y_train mean: {Y_train.mean().item():.4f}, std: {Y_train.std().item():.4f}")


Train: torch.Size([69, 12, 2, 201, 281]) torch.Size([69, 6, 2, 201, 281])
Val: torch.Size([11, 12, 2, 201, 281]) torch.Size([11, 6, 2, 201, 281])
Test: torch.Size([13, 12, 2, 201, 281]) torch.Size([13, 6, 2, 201, 281])


In [37]:
import torch
print("Torch version:", torch.__version__)
print("Compiled CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())



Torch version: 2.9.1+cpu
Compiled CUDA: None
CUDA available: False


In [ ]:
#Create DataLoaders
from torch.utils.data import TensorDataset

BATCH_SIZE = 2  # keep small for memory

train_loader = DataLoader(
    TensorDataset(X_train, Y_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, Y_val),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test, Y_test),
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [ ]:
#Initialize Predrnn model with increased capacity

model = PredRNN(
    input_dim=2,                  # u, v
    hidden_dims=[128, 128, 128, 128],  # 4-layer PredRNN with larger hidden dims
    kernel_size=3,
    num_layers=4,
    out_steps=Y_train.shape[1]     # Tout
).to(DEVICE)

print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")


1.075138 M parameters


In [ ]:
#train predrnn model with more epochs
EPOCHS = 50  # Increased from 20 to 50 for better learning
LR = 1e-3

train_predrnn(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR
)



Epoch 1/20 | TF ratio: 0.05 | Train Loss: 0.5929 | Val Loss: 0.5397
Epoch 2/20 | TF ratio: 0.10 | Train Loss: 0.3512 | Val Loss: 0.3006


KeyboardInterrupt: 

In [ ]:
# Evaluate on test set
model.eval()
criterion = torch.nn.MSELoss()
test_loss = 0

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        preds = model(x, teacher_forcing_ratio=0.0)
        test_loss += criterion(preds, y).item()

test_loss /= len(test_loader)
print("PredRNN Test MSE:", test_loss)

In [ ]:
# Quick visualization: u and v predicted vs target
model.eval()
with torch.no_grad():
    x, y = next(iter(test_loader))
    x, y = x.to(DEVICE), y.to(DEVICE)
    preds = model(x, teacher_forcing_ratio=0.0).cpu().numpy()
    targets = y.cpu().numpy()

# Denormalize
mean = np.load("data/processed/mean.npy").reshape(1, 1, 2, 1, 1)
std = np.load("data/processed/std.npy").reshape(1, 1, 2, 1, 1)
preds = preds * std + mean
targets = targets * std + mean

# Plot u and v time series at a specific location
sample, lat_idx, lon_idx = 0, 100, 140
time_steps = range(preds.shape[1])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# U component
ax1.plot(time_steps, targets[sample, :, 0, lat_idx, lon_idx], 'o-', label='Target', linewidth=2)
ax1.plot(time_steps, preds[sample, :, 0, lat_idx, lon_idx], 's--', label='Predicted', linewidth=2)
ax1.set_xlabel('Time Step')
ax1.set_ylabel('U Wind (m/s)')
ax1.set_title('U Component')
ax1.legend()
ax1.grid(True, alpha=0.3)

# V component
ax2.plot(time_steps, targets[sample, :, 1, lat_idx, lon_idx], 'o-', label='Target', linewidth=2, color='green')
ax2.plot(time_steps, preds[sample, :, 1, lat_idx, lon_idx], 's--', label='Predicted', linewidth=2, color='orange')
ax2.set_xlabel('Time Step')
ax2.set_ylabel('V Wind (m/s)')
ax2.set_title('V Component')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
